# TP 2 - Administrateur HDFS : organiser et contrôler un espace de données

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syhard87/tp_hadloop/blob/claude/tp-hadoop-avances-partie-a-mwgfq7/notebooks/TP2_Administrateur_HDFS.ipynb)

Source : `sources/TP_Hadoop_avances_avec_TP_ecosysteme.pdf`, pages 12 et 13.

**Contexte** : on administre l'espace HDFS d'une entreprise. On crée une organisation `raw/curated/analytics/rejects`,
on charge les données et on contrôle leur stockage.

**Jeu de données** : uniquement `logs_web.csv`, fourni par l'enseignant (200 lignes : timestamp, ip, page, statut).

**Mode d'emploi** : vérifier que l'environnement est connecté (coche verte en haut à droite), puis exécuter les cellules
**dans l'ordre**. Si la session redémarre, reprendre depuis la cellule 1.

## 1. Récupérer le dépôt (jeu de données + script d'installation)
Après cette cellule, le panneau **Fichiers** (🔄 pour rafraîchir) affiche `tp_hadloop/data/logs_web.csv`.

In [ ]:
# Supprime le dossier d'exemples fourni par défaut par Google Colab (inutile pour le TP)
!rm -rf /content/sample_data
!rm -rf /content/tp_hadloop
!git clone -q --depth 1 -b claude/tp-hadoop-avances-partie-a-mwgfq7 https://github.com/Syhard87/tp_hadloop.git /content/tp_hadloop
!ls -l /content/tp_hadloop/data /content/tp_hadloop/scripts

## 2. Variables d'environnement et installation de Hadoop (HDFS + YARN)
Le script `scripts/setup_hadoop_colab.sh` reprend les cellules 1 à 4 du TP 1, avec la correction pour `root`.
Il ajoute YARN, nécessaire pour `yarn node -list`.
Il peut être relancé sans risque : il ne reformate jamais un NameNode déjà formaté.
**Durée** : 2 à 4 minutes à la première exécution.

In [ ]:
import os
HADOOP_HOME = "/content/hadoop-3.5.0"
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
if f"{HADOOP_HOME}/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]
os.makedirs("/content/preuves_tp2", exist_ok=True)

!bash /content/tp_hadloop/scripts/setup_hadoop_colab.sh

## 3. Le jeu de données `logs_web.csv`
Copie dans `/content/tp2` (le chemin **local**), puis aperçu du contenu et de l'anomalie attendue (plusieurs statuts 404).

In [ ]:
!mkdir -p /content/tp2 && cp /content/tp_hadloop/data/logs_web.csv /content/tp2/
!ls -l /content/tp2 && head -n 5 /content/tp2/logs_web.csv

import pandas as pd
logs = pd.read_csv("/content/tp2/logs_web.csv")
print("\nLignes de données :", len(logs), "| IP distinctes :", logs["ip"].nunique(),
      "| période :", logs["timestamp"].min(), "->", logs["timestamp"].max())
print("\nRépartition des statuts :")
print(logs["statut"].value_counts().to_string())
print("\nPages en erreur 404 :")
print(logs[logs["statut"] == 404]["page"].value_counts().to_string())

## Mission A - Préparer les services
**Attendu** : `jps` affiche NameNode, DataNode, SecondaryNameNode, ResourceManager et NodeManager ;
`Live datanodes (1)` ; YARN indique `Total Nodes:1` avec un nœud `RUNNING`.

In [ ]:
!jps | tee /content/preuves_tp2/A_jps.txt
!hdfs dfsadmin -report 2>/dev/null | tee /content/preuves_tp2/A_dfsadmin_report.txt
!yarn node -list 2>/dev/null | tee /content/preuves_tp2/A_yarn_nodes.txt

## Mission B - Construire l'arborescence
L'arborescence est celle de l'énoncé. Seul `web_logs` reçoit des données dans ce TP. Les dossiers `sales`, `customers`
et `products` sont créés pour respecter l'organisation demandée, mais restent vides.

In [ ]:
%%bash
D=ingestion_date=2026-10-11
hdfs dfs -mkdir -p \
  /data/raw/sales/$D \
  /data/raw/customers/$D \
  /data/raw/products/$D \
  /data/raw/web_logs/$D \
  /data/curated/sales/year=2026/month=10 \
  /data/analytics \
  /data/rejects
hdfs dfs -ls -R /data | tee /content/preuves_tp2/B_arborescence.txt

## Mission C - Charger et inspecter

In [ ]:
%%bash
D=ingestion_date=2026-10-11
# 10. Charger le fichier dans la zone raw
hdfs dfs -put -f /content/tp2/logs_web.csv /data/raw/web_logs/$D/

# 11. Vérifier sa présence et lire un extrait
# (le message « Unable to write to output stream » est normal : head ferme le flux)
hdfs dfs -ls -R /data/raw/web_logs | tee /content/preuves_tp2/C_ls.txt
echo
hdfs dfs -cat /data/raw/web_logs/$D/logs_web.csv 2>/dev/null | head -n 5

# 12. Taille logique (1re colonne) et espace consommé avec la réplication (2e colonne)
echo; echo "=== du -h /data/raw ==="
hdfs dfs -du -h /data/raw | tee /content/preuves_tp2/C_du.txt

### Isoler les lignes 404 dans `rejects`
L'énoncé demande la sortie `fsck` d'**au moins deux fichiers**. Avec un seul jeu de données, on produit un second fichier
utile : les lignes en **404** (l'anomalie du jeu de données). On les lit **depuis HDFS** et on les dépose dans la zone
`rejects`, ce qui illustre aussi le rôle de cette zone.

In [ ]:
%%bash
D=ingestion_date=2026-10-11
hdfs dfs -mkdir -p /data/rejects/web_logs/$D
hdfs dfs -cat /data/raw/web_logs/$D/logs_web.csv \
  | awk -F, 'NR==1 || $4==404' \
  | hdfs dfs -put -f - /data/rejects/web_logs/$D/logs_404.csv
echo "Lignes 404 isolées (hors en-tête) : $(hdfs dfs -cat /data/rejects/web_logs/$D/logs_404.csv | tail -n +2 | wc -l)"
hdfs dfs -cat /data/rejects/web_logs/$D/logs_404.csv 2>/dev/null | head -n 4

### 13. Blocs et réplication (fsck sur les deux fichiers)

In [ ]:
%%bash
D=ingestion_date=2026-10-11
for f in /data/raw/web_logs/$D/logs_web.csv /data/rejects/web_logs/$D/logs_404.csv; do
  hdfs fsck $f -files -blocks -locations 2>/dev/null | sed '/^Erasure Coded/,$d'
done | tee /content/preuves_tp2/C_fsck.txt

### 14. Tableau comparatif local / HDFS

In [ ]:
import subprocess, os, math, pandas as pd

D = "ingestion_date=2026-10-11"
hdfs_path = f"/data/raw/web_logs/{D}/logs_web.csv"
local = "/content/tp2/logs_web.csv"

def hdfs_stat(path):
    out = subprocess.run(["hdfs", "dfs", "-stat", "%b %r %o", path], capture_output=True, text=True).stdout.split()
    return int(out[0]), int(out[1]), int(out[2])

taille, repl, bloc = hdfs_stat(hdfs_path)
tableau = pd.DataFrame({
    "Local": [local, "Disque de la VM Colab (Linux)", os.path.getsize(local), "-", "-",
              sum(1 for _ in open(local)), "ls, cat, du"],
    "HDFS": [hdfs_path, "HDFS (hdfs://localhost:9000)", taille, repl, f"{bloc} ({math.ceil(taille / bloc)} bloc)",
             int(subprocess.run(f"hdfs dfs -cat {hdfs_path} | wc -l", shell=True, capture_output=True, text=True).stdout),
             "hdfs dfs -ls, -cat, -du"],
}, index=["Chemin", "Système de fichiers", "Taille (octets)", "Réplication", "Taille de bloc", "Lignes", "Commandes"])
tableau.to_csv("/content/preuves_tp2/C_tableau_local_hdfs.csv")
tableau

## Mission D - Droits et cycle de vie
On produit d'abord un petit résultat dans `analytics` (nombre de requêtes par statut), puis on pose le marqueur `_READY`.

In [ ]:
%%bash
D=ingestion_date=2026-10-11
# Résultat analytique : nombre de requêtes par statut HTTP
hdfs dfs -mkdir -p /data/analytics/web_logs
hdfs dfs -cat /data/raw/web_logs/$D/logs_web.csv \
  | awk -F, 'NR>1 {n[$4]++} END {print "statut,nb_requetes"; for (s in n) print s "," n[s]}' \
  | hdfs dfs -put -f - /data/analytics/web_logs/statuts.csv
hdfs dfs -cat /data/analytics/web_logs/statuts.csv

# Droits et marqueur
hdfs dfs -chmod 750 /data/curated
hdfs dfs -chmod 770 /data/analytics
hdfs dfs -touchz /data/analytics/_READY

echo
hdfs dfs -ls -d /data/raw /data/curated /data/analytics /data/rejects | tee /content/preuves_tp2/D_droits.txt
echo
hdfs dfs -stat "%n %b %r" /data/analytics/_READY | tee -a /content/preuves_tp2/D_droits.txt
hdfs dfs -stat "%n %b %r" /data/raw/web_logs/$D/logs_web.csv | tee -a /content/preuves_tp2/D_droits.txt
hdfs dfs -test -e /data/analytics/_READY && echo "_READY présent : les données analytics peuvent être consommées"

## Arborescence finale et sauvegarde des preuves dans l'espace partagé (Drive)
Crée `MyDrive/TP_Hadoop_partage/preuves_tp2`. Partager ce dossier depuis Drive (clic droit → *Partager*).

In [ ]:
!hdfs dfs -ls -R /data | tee /content/preuves_tp2/D_arborescence_finale.txt
from google.colab import drive
import shutil
drive.mount('/content/drive')
dest = "/content/drive/MyDrive/TP_Hadoop_partage/preuves_tp2"
shutil.copytree("/content/preuves_tp2", dest, dirs_exist_ok=True)
print("Preuves copiées dans", dest, ":", sorted(os.listdir(dest)))

---
# Réponses

## Mission A
- **Présence des services** : `jps` affiche **NameNode**, **DataNode** et **SecondaryNameNode** pour HDFS,
  ainsi que **ResourceManager** et **NodeManager** pour YARN.
- **DataNodes vivants** : `Live datanodes (1)`, `Dead datanodes (0)`. En mode pseudo-distribué, un seul DataNode tourne sur la machine Colab.
- **HDFS ou YARN ?** **HDFS** est la couche de **stockage**. Le NameNode garde les métadonnées : arborescence, droits,
  liste des blocs et emplacement de chaque bloc. Les DataNodes stockent physiquement les blocs. **YARN** est la couche de
  **gestion des ressources et d'exécution des traitements**. Le ResourceManager alloue le CPU et la mémoire aux applications,
  et les NodeManagers lancent les conteneurs sur chaque nœud. HDFS répond à « où sont les données ? », YARN à « qui calcule,
  avec quelles ressources ? ». On peut lire et écrire dans HDFS sans YARN, mais un job MapReduce ou Spark sur le cluster a besoin de YARN.

## Mission B
| Zone | Rôle | Dans ce TP |
|---|---|---|
| `raw/` | Données **brutes**, telles que reçues, jamais modifiées. On peut toujours rejouer un traitement ou auditer une donnée. | `logs_web.csv` tel que livré |
| `curated/` | Données **nettoyées et validées**, avec un partitionnement métier (`year=/month=`). C'est la source fiable des traitements. | Préparé, vide |
| `analytics/` | **Résultats** prêts à consommer (agrégats, indicateurs) pour les analystes et les tableaux de bord. | `statuts.csv` (requêtes par statut) |
| `rejects/` | Lignes **rejetées** ou en anomalie, isolées pour ne pas polluer `curated` et rester analysables. | `logs_404.csv` (48 lignes 404) |

**Rôle de `ingestion_date`** dans `raw` : c'est une **partition technique** qui indique **le jour de réception** des
données, pas leur date métier. Elle permet :
- de rejouer ou supprimer un chargement précis sans toucher aux autres ;
- d'éviter qu'une nouvelle livraison écrase la précédente ;
- de tracer l'origine d'une donnée ;
- d'appliquer une règle de rétention par date.

La forme `cle=valeur` est reconnue par Hive et Spark comme une colonne de partition.

## Mission C
- **10-11** : `logs_web.csv` est présent dans `/data/raw/web_logs/ingestion_date=2026-10-11/`, et `hdfs dfs -cat … | head`
  affiche l'en-tête `timestamp,ip,page,statut` suivi des premières lignes.
- **12. Taille logique** : **9 766 octets** (environ 9,5 K). Dans `du`, la 1re colonne donne la taille logique et la 2e
  l'espace consommé en tenant compte de la réplication. Ici les deux sont égales, car la réplication vaut 1.
- **13. Blocs et réplication** : `fsck` affiche `1 block(s)`, `len=9766 Live_repl=1`, `replication=1`, statut `HEALTHY`.
  Le fichier `logs_404.csv` (2 522 octets) tient aussi dans 1 bloc. Un bloc HDFS n'occupe sur disque que la taille réelle
  des données : un fichier de 9,5 Ko ne consomme pas 128 Mo. En revanche, chaque fichier et chaque bloc coûtent une
  entrée en mémoire dans le NameNode, d'où le problème des petits fichiers (TP 4).
- **14. Local ou HDFS** :

| | Local | HDFS |
|---|---|---|
| Chemin | `/content/tp2/logs_web.csv` | `/data/raw/web_logs/ingestion_date=2026-10-11/logs_web.csv` |
| Système de fichiers | Disque de la VM Colab | Distribué (`hdfs://localhost:9000`) |
| Commandes | `ls`, `cat`, `du` | `hdfs dfs -ls`, `-cat`, `-du` |
| Stockage | Un fichier sur un disque | Découpé en blocs, répliqué, métadonnées dans le NameNode |
| Taille | 9 766 o, 201 lignes | 9 766 o, 201 lignes (identique) |

Le contenu est identique, mais le fichier HDFS n'est pas visible avec un `ls` Linux. Ses blocs sont stockés sous
`/content/hdfs/data` sous forme de fichiers `blk_…`.

**Ce que révèle le jeu de données** : 146 requêtes en 200, **48 en 404** (24 `/ancienne-page` et 24 `/produits/xyz`,
soit 24 % du trafic), 5 redirections 301 et 1 erreur serveur 500 sur `/connexion`. Les 404 portent toutes sur deux URL :
il s'agit de liens cassés à corriger, ou à rediriger en 301.

## Mission D
- **750 sur `/data/curated`** (`drwxr-x---`) : le **propriétaire** a lecture, écriture et parcours (7). Le **groupe** peut lire
  et parcourir, mais pas écrire (5). Les **autres** n'ont aucun accès (0). Seul le pipeline propriétaire écrit les données
  validées ; l'équipe peut les lire sans risquer de les modifier.
- **770 sur `/data/analytics`** (`drwxrwx---`) : le propriétaire **et le groupe** ont tous les droits (7). Les autres n'ont
  aucun accès. C'est un espace de travail collaboratif : les analystes du groupe créent et suppriment leurs résultats.
- **Fichier marqueur `_READY`** : c'est un fichier vide (0 octet) écrit **en dernier**, une fois toutes les données produites
  (ici après `statuts.csv`). Les consommateurs (job aval, ordonnanceur, tableau de bord) attendent sa présence
  (`hdfs dfs -test -e …/_READY`) avant de lire. Cela évite de lire un dossier incomplet pendant l'écriture.
  Le principe est le même que `_SUCCESS`, écrit par MapReduce et Spark.
- **Règle d'archivage ou de suppression de `raw`** :
  1. Conserver les partitions `ingestion_date` des **30 derniers jours** dans `raw`.
  2. Au-delà, une fois la partition traitée vers `curated` et `_READY` présent, l'**archiver** :
     compression, archive HAR ou stockage froid avec `hdfs storagepolicies -setStoragePolicy -policy COLD`.
  3. **Supprimer** les archives après la durée légale de conservation. Pour des logs web contenant des adresses IP
     (données personnelles au sens du RGPD), on peut retenir par exemple 1 an.

  Ne jamais supprimer une partition `raw` qui n'a pas encore été traitée.